# Extract KG Triples from `real_contradiction` + sampled negatives

Open-extraction pipeline over contradiction pairs from the 2008 paper:

1. **Open extraction**: LLM picks predicates freely; we canonicalize post-hoc.
2. **Hallucination pruning** (van Cauter & Yakovets, 2024, §3.3): drop any triple whose subject or object is absent from the source sentence (token-exact, case-folded).
3. **Negative pairs**: sample 131 NO-contradiction rows from `contradiction_binary.csv` and run the same extraction + pruning so the downstream graph contains balanced YES / NO claims.

Outputs are saved to `data/processed/kg_triples/`.

In [81]:
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from openai import OpenAI
from pydantic_settings import BaseSettings, SettingsConfigDict
from tqdm import tqdm

In [82]:
class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",
    )
    openai_api_key: str
    llm_model: str = "gpt-5.4-mini"


settings = Settings()
client = OpenAI(api_key=settings.openai_api_key)
OUTPUT_DIR = Path("data/processed/kg_triples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using model: {settings.llm_model}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Using model: gpt-5.4-mini
Output directory: D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\experiments\data\processed\kg_triples


In [83]:
df = pd.read_csv("data/processed/finding_contradictions_in_text_2008/real_contradiction.csv")
print(f"Loaded {len(df)} pairs")
df.head()

Loaded 131 pairs


,contradiction,type,t,h
0,YES,negation,Tariq Aziz was not considered a member of Sadd...,Tariq Aziz was in Saddam's inner circle.
1,YES,lexical,Tariq Aziz kept outside the closed circle of S...,Tariq Aziz was in Saddam's inner circle.
2,YES,negation,Tariq Aziz was not one of the most powerful fi...,Tariq Aziz was prominent in the regime.
3,YES,lexical,Tariq Aziz retained influence.,Aziz had virtually no power.
4,YES,numeric,Alexandros Yiotopoulos is 58.,Alexandros Yiotopoulos is 59 years old.


## Shared extraction helpers

`extract_triples()` calls OpenAI in JSON mode with a supplied prompt. `canonicalize()` normalizes predicate surface forms for the closed schema.

In [84]:
OPEN_PROMPT = (
    "Extract knowledge graph triples from the following sentence.\n\n"
    "Rules:\n"
    "- Extract multiple triples only if the sentence expresses multiple distinct facts. "
    "Do NOT artificially split one fact into many.\n"
    "- Every NUMERIC quantity (duration, date, age, count, distance, year, time span) IS "
    "a distinct fact and MUST become its own triple, even when it lives in a subordinate "
    "adverbial clause such as 'After X...', 'Before X...', 'During X...', or 'For X...'. "
    "Do NOT drop numbers.\n"
    '    Sentence: "After spending about two hours in the sky, he concluded Y."\n'
    "    GOOD: [(he, spent_in_sky_for, two hours),\n"
    "           (he, concluded, Y)]\n"
    "    BAD (drops the duration):\n"
    "         [(he, concluded, Y)]\n"
    "- Subjects and objects must be SPECIFIC named entities. Keep the modifiers that "
    "make them unique (e.g., 'Saddam's inner circle', NOT just 'circle'; "
    "'Sunni Moslem cronies', NOT just 'cronies'). Strip only leading articles (a/an/the).\n"
    "- Reuse the same surface form across triples for the same entity. If you name "
    "something 'Saddam's inner circle' once, refer to it the same way in any later triple.\n"
    "- Example of good extraction:\n"
    '    Sentence: "Tariq Aziz kept outside Saddam\'s inner circle of Sunni cronies."\n'
    "    GOOD: [(Tariq Aziz, kept_outside, Saddam's inner circle),\n"
    "           (Saddam's inner circle, composed_of, Sunni cronies)]\n"
    "    BAD (loses specificity, creates generic hub entities):\n"
    "         [(Tariq Aziz, kept_outside, circle),\n"
    "          (circle, belongs_to, Saddam),\n"
    "          (circle, described_as, Sunni cronies)]\n"
    "- Preserve negation inside the predicate. If a claim is explicitly negated "
    "(not, never, no, refused to, failed to, etc.), prefix the predicate with 'not '. "
    "Do NOT bury negation in the object.\n\n"
    "Return strict JSON of the form: "
    '{{"triples": [{{"subject": "...", "predicate": "...", "object": "..."}}]}}\n\n'
    "Sentence: {sentence}"
)


def extract_triples(sentence: str, prompt_template: str, **kwargs) -> list[dict]:
    prompt = prompt_template.format(sentence=sentence, **kwargs)
    try:
        response = client.chat.completions.create(
            model=settings.llm_model,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.0,
        )
        payload = json.loads(response.choices[0].message.content)
        triples = payload.get("triples", [])
        return [t for t in triples if isinstance(t, dict)]
    except Exception as e:
        print(f"Extraction failed for sentence: {sentence[:60]}... | {e}")
        return []


def canonicalize(pred: str | None) -> str | None:
    if pred is None:
        return None
    return pred.strip().lower().replace(" ", "_").replace("-", "_")

## Smoke test on first 3 pairs (open extraction)

Confirms the API call and JSON parsing work before committing to 131 pairs.

In [85]:
for idx, row in enumerate(df.head(3).itertuples(index=False), start=1):
    print(f"[id={idx}] type={row.type}")
    print(f"  t: {row.t}")
    print(f"  h: {row.h}")
    open_t = [{**tr, "predicate": canonicalize(tr.get("predicate"))} for tr in extract_triples(row.t, OPEN_PROMPT)]
    open_h = [{**tr, "predicate": canonicalize(tr.get("predicate"))} for tr in extract_triples(row.h, OPEN_PROMPT)]
    print(f"  open t: {open_t}")
    print(f"  open h: {open_h}")
    print("─" * 100)

[id=1] type=negation
  t: Tariq Aziz was not considered a member of Saddam's innermost circle.
  h: Tariq Aziz was in Saddam's inner circle.
  open t: [{'subject': 'Tariq Aziz', 'predicate': 'not_considered_member_of', 'object': "Saddam's innermost circle"}]
  open h: [{'subject': 'Tariq Aziz', 'predicate': 'was_in', 'object': "Saddam's inner circle"}]
────────────────────────────────────────────────────────────────────────────────────────────────────
[id=2] type=lexical
  t: Tariq Aziz kept outside the closed circle of Saddam's Sunni Moslem cronies.
  h: Tariq Aziz was in Saddam's inner circle.
  open t: [{'subject': 'Tariq Aziz', 'predicate': 'kept_outside', 'object': "the closed circle of Saddam's Sunni Moslem cronies"}]
  open h: [{'subject': 'Tariq Aziz', 'predicate': 'was_in', 'object': "Saddam's inner circle"}]
────────────────────────────────────────────────────────────────────────────────────────────────────
[id=3] type=negation
  t: Tariq Aziz was not one of the most powerful

## Open extraction on all 131 pairs

Appends to JSONL after each pair so a partial run doesn't lose work.

In [86]:
open_file = OUTPUT_DIR / "open_extraction.jsonl"

with open_file.open("w", encoding="utf-8") as f:
    for idx, row in enumerate(tqdm(list(df.itertuples(index=False)), desc="Open"), start=1):
        t_raw = extract_triples(row.t, OPEN_PROMPT)
        h_raw = extract_triples(row.h, OPEN_PROMPT)
        t_triples = [{**tr, "predicate": canonicalize(tr.get("predicate"))} for tr in t_raw]
        h_triples = [{**tr, "predicate": canonicalize(tr.get("predicate"))} for tr in h_raw]
        record = {
            "id": str(idx),
            "type": row.type,
            "t_text": row.t,
            "h_text": row.h,
            "t_triples": t_triples,
            "h_triples": h_triples,
            "notes": "",
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

print(f"Open extraction done -> {open_file}")

Open: 100%|██████████| 131/131 [05:11<00:00,  2.38s/it]

Open extraction done -> data\processed\kg_triples\open_extraction.jsonl


## Hallucination pruning (van Cauter & Yakovets, 2024, §3.3)

Drop any triple whose subject or object does not appear as a contiguous run of tokens in the source sentence (token-exact, case-folded; `"filter"` does not match `"filters"`). No predicate filter - any predicate is allowed. Writes `open_extraction_pruned.jsonl`.

In [87]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


def appears_in_tokens(span: str | None, source_tokens: list[str]) -> bool:
    if not span:
        return False
    span_tokens = tokenize(span)
    if not span_tokens:
        return False
    n = len(span_tokens)
    for i in range(len(source_tokens) - n + 1):
        if source_tokens[i : i + n] == span_tokens:
            return True
    return False


def prune_triples_open(triples: list[dict], source: str) -> list[dict]:
    source_tokens = tokenize(source)
    kept = []
    for tr in triples:
        if not tr.get("predicate"):
            continue
        if not appears_in_tokens(tr.get("subject"), source_tokens):
            continue
        if not appears_in_tokens(tr.get("object"), source_tokens):
            continue
        kept.append(tr)
    return kept


def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


open_file = OUTPUT_DIR / "open_extraction.jsonl"
open_pruned_file = OUTPUT_DIR / "open_extraction_pruned.jsonl"

raw_open = load_jsonl(open_file)
before_open = sum(len(r["t_triples"]) + len(r["h_triples"]) for r in raw_open)

with open_pruned_file.open("w", encoding="utf-8") as f:
    for r in raw_open:
        r_pruned = {
            **r,
            "t_triples": prune_triples_open(r["t_triples"], r["t_text"]),
            "h_triples": prune_triples_open(r["h_triples"], r["h_text"]),
        }
        f.write(json.dumps(r_pruned, ensure_ascii=False) + "\n")

open_pruned = load_jsonl(open_pruned_file)
after_open = sum(len(r["t_triples"]) + len(r["h_triples"]) for r in open_pruned)
dropped_open = before_open - after_open

print(f"Open triples before pruning: {before_open}")
print(f"Open triples after pruning:  {after_open}")
if before_open:
    print(f"Dropped: {dropped_open} ({100 * dropped_open / before_open:.1f}% of original)")
print(f"Pruned output -> {open_pruned_file}")

Open triples before pruning: 689
Open triples after pruning:  652
Dropped: 37 (5.4% of original)
Pruned output -> data\processed\kg_triples\open_extraction_pruned.jsonl


## Negative pairs: all 448 NO rows from `contradiction_binary.csv`

Same OPEN_PROMPT + pruning as the positive side. IDs are prefixed `cbno_1`..`cbno_448` so they do not collide with real_contradiction IDs (`1`..`131`).

Writes `binary_no_extraction.jsonl` and `binary_no_extraction_pruned.jsonl`.

In [92]:
binary_df = pd.read_csv("data/processed/finding_contradictions_in_text_2008/contradiction_binary.csv")
no_df = binary_df[binary_df["contradiction"] == "NO"].reset_index(drop=True)
print(f"Total NO pairs: {len(no_df)}")
no_df.head()

Total NO pairs: 448


,contradiction,t,h
0,NO,"Meanwhile, in an exclusive interview with a TI...","Ahmedinejad was never attacked by the US, Fran..."
1,NO,"From Joss Stone to Keith Richards to Sting, th...",Eric Clapton is not in business with Les Paul.
2,NO,Hinostroza's children told police that four ho...,Hinostroza's children never attaked their moth...
3,NO,"Alan Mulally, Boeing's head of the unit, said ...",Boeing will never be a subsidiary of Airbus SAS.
4,NO,Thanks to the recent acquisition of J.D. Edwar...,J.D.E. will never be the owner of Oracle.


In [93]:
binary_no_file = OUTPUT_DIR / "binary_no_extraction.jsonl"

with binary_no_file.open("w", encoding="utf-8") as f:
    for idx, row in enumerate(tqdm(list(no_df.itertuples(index=False)), desc="Binary NO"), start=1):
        t_raw = extract_triples(row.t, OPEN_PROMPT)
        h_raw = extract_triples(row.h, OPEN_PROMPT)
        t_triples = [{**tr, "predicate": canonicalize(tr.get("predicate"))} for tr in t_raw]
        h_triples = [{**tr, "predicate": canonicalize(tr.get("predicate"))} for tr in h_raw]
        record = {
            "id": f"cbno_{idx}",
            "type": "none",
            "contradiction_label": "NO",
            "t_text": row.t,
            "h_text": row.h,
            "t_triples": t_triples,
            "h_triples": h_triples,
            "notes": "",
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

print(f"Binary NO extraction done -> {binary_no_file}")

Binary NO: 100%|██████████| 448/448 [18:59<00:00,  2.54s/it]

Binary NO extraction done -> data\processed\kg_triples\binary_no_extraction.jsonl


In [94]:
binary_no_pruned_file = OUTPUT_DIR / "binary_no_extraction_pruned.jsonl"
raw_binary_no = load_jsonl(binary_no_file)
before_neg = sum(len(r["t_triples"]) + len(r["h_triples"]) for r in raw_binary_no)

with binary_no_pruned_file.open("w", encoding="utf-8") as f:
    for r in raw_binary_no:
        r_pruned = {
            **r,
            "t_triples": prune_triples_open(r["t_triples"], r["t_text"]),
            "h_triples": prune_triples_open(r["h_triples"], r["h_text"]),
        }
        f.write(json.dumps(r_pruned, ensure_ascii=False) + "\n")

binary_no_pruned = load_jsonl(binary_no_pruned_file)
after_neg = sum(len(r["t_triples"]) + len(r["h_triples"]) for r in binary_no_pruned)
dropped_neg = before_neg - after_neg

print(f"Binary NO triples before pruning: {before_neg}")
print(f"Binary NO triples after pruning:  {after_neg}")
if before_neg:
    print(f"Dropped: {dropped_neg} ({100 * dropped_neg / before_neg:.1f}% of original)")
print(f"Pruned output -> {binary_no_pruned_file}")

Binary NO triples before pruning: 2143
Binary NO triples after pruning:  2047
Dropped: 96 (4.5% of original)
Pruned output -> data\processed\kg_triples\binary_no_extraction_pruned.jsonl


## Summary comparison

In [95]:
def coverage(records: list[dict]) -> dict:
    both = sum(1 for r in records if r["t_triples"] and r["h_triples"])
    t_has = sum(1 for r in records if r["t_triples"])
    h_has = sum(1 for r in records if r["h_triples"])
    return {"total": len(records), "t_has_triples": t_has, "h_has_triples": h_has, "both": both}


def count_triples(records: list[dict]) -> int:
    return sum(len(r["t_triples"]) + len(r["h_triples"]) for r in records)


def count_predicates(records: list[dict]) -> Counter:
    return Counter(tr.get("predicate") for r in records for tr in r["t_triples"] + r["h_triples"])


open_records = load_jsonl(open_file)
open_pruned = load_jsonl(open_pruned_file)
binary_no = load_jsonl(binary_no_file)
binary_no_pruned = load_jsonl(binary_no_pruned_file)

summary_rows = [
    ("Open (YES)", open_records),
    ("Open pruned (YES)", open_pruned),
    ("Binary NO", binary_no),
    ("Binary NO pruned", binary_no_pruned),
]

print(f"{'Pipeline':22} {'Pairs':>6} {'t_has':>6} {'h_has':>6} {'both':>6} {'Triples':>9}")
for name, recs in summary_rows:
    cov = coverage(recs)
    print(f"{name:22} {cov['total']:>6} {cov['t_has_triples']:>6} {cov['h_has_triples']:>6} {cov['both']:>6} {count_triples(recs):>9}")

print()
print("Open pruned (YES) top predicates:")
for pred, count in count_predicates(open_pruned).most_common(15):
    print(f"  {pred}: {count}")

print()
print("Binary NO pruned top predicates:")
for pred, count in count_predicates(binary_no_pruned).most_common(15):
    print(f"  {pred}: {count}")

Pipeline                Pairs  t_has  h_has   both   Triples
Open (YES)                131    131    131    131       689
Open pruned (YES)         131    127    126    124       652
Binary NO                 448    448    448    448      2143
Binary NO pruned          448    445    424    421      2047

Open pruned (YES) top predicates:
  said: 26
  is: 23
  was: 12
  killed: 10
  said_in: 7
  include: 7
  refused_to_allow: 6
  caused: 6
  found: 6
  has_been_arrested_in: 5
  allowed: 5
  told: 5
  used: 5
  has_tense: 5
  is_as_adamant_as: 4

Binary NO pruned top predicates:
  is: 54
  said: 36
  include: 23
  includes: 22
  are: 19
  not_is: 17
  occurred_on: 14
  killed: 12
  located_in: 11
  occurred_in: 11
  was: 10
  had: 10
  not_was: 8
  not_is_a: 8
  says: 8
